# LDA Modelling


## 1. Setup

In [1]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns

import os
import joblib
from tqdm.auto import tqdm


In [2]:
RANDOM_STATE = 42

DATA_PATH = '../data/processed_data/reviews_clean.parquet'
MODELS_DIR = '../models/sentiment'

os.makedirs(MODELS_DIR, exist_ok = True)

In [3]:
VECT_PARAMS = dict(
    ngram_range = (1, 2), # keep bigrams -> 'not good' stays one feature
    min_df = 5,          # lower than 03; keep rare-but-strong words 
    token_pattern = r"(?u)\b\w[\w']+\b", # keep contractions whole -> won't != won
    lowercase = False, # review_clean is already lowercased in 02
)


# Splitting group aware and stratified 
N_SPLITS = 5 # first fold -> 80/20 hold out test

### 2. Load Dataset

In [4]:
data = pd.read_parquet(DATA_PATH)
data.head(10)

,funny,helpful,hour_played,is_early_access_review,title,label,review_clean,language,duplicates,word_count
0,2.0,4,578,False,Expansion - Hearts of Iron IV: Man the Guns,1,> played as german reich> declare war on belgi...,EN,False,31
1,0.0,0,184,False,Expansion - Hearts of Iron IV: Man the Guns,1,yes.,EN,False,1
2,0.0,0,892,False,Expansion - Hearts of Iron IV: Man the Guns,1,very good game although a bit overpriced in my...,EN,False,29
3,126.0,1086,676,False,Dead by Daylight,1,out of all the reviews i wrote this one is pro...,EN,False,419
4,85.0,2139,612,False,Dead by Daylight,1,disclaimer i survivor main. i play games for f...,EN,False,273
5,4.0,55,2694,False,Dead by Daylight,1,english after playing for more than two years ...,EN,False,819
6,12.0,228,48,False,Dead by Daylight,1,out of all the reviews i wrote this one is pro...,EN,True,419
7,295.0,219,71,False,Dead by Daylight,1,i have never been told to kill myself more tha...,EN,False,14
8,2.0,54,400,False,Dead by Daylight,1,any longtime dead by daylight player knows tha...,EN,False,339
9,380.0,271,414,False,Dead by Daylight,1,if you think cs go is toxic try this game,EN,False,10


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 406781 entries, 0 to 406780
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   funny                   406731 non-null  float64
 1   helpful                 406781 non-null  int64  
 2   hour_played             406781 non-null  int64  
 3   is_early_access_review  406781 non-null  bool   
 4   title                   406781 non-null  str    
 5   label                   406781 non-null  int64  
 6   review_clean            406781 non-null  str    
 7   language                406781 non-null  str    
 8   duplicates              406781 non-null  bool   
 9   word_count              406781 non-null  int64  
dtypes: bool(2), float64(1), int64(4), str(3)
memory usage: 123.3 MB


In [6]:
print(f"NaN values in review_clean: {data['review_clean'].isnull().sum()}")
n_empty = (data['review_clean'].str.strip() == '').sum()#
print(f'Empty review_clean strings: {n_empty}')
print(f"NaN values in label: {data['label'].isnull().sum()}")

NaN values in review_clean: 0
Empty review_clean strings: 0
NaN values in label: 0


## 3. Scope & Decisions

Notebook 03 tuned preprocessing for clean topic clusters. Notebook 04 tunes it
to separate positive reviews from negative ones. The table records where the
conservative variant diverges from the aggressive one in 03, and why.

| Step | Decision | Rationale |
|---|---|---|
| Negations | Keep `not`, `no`, `never`, contractions | They mark the strongest negative signal. Removing them flips a review's meaning. |
| n-grams | `ngram_range=(1, 2)` | A bigram keeps `not good` as one feature. Unigrams split it and lose the link. |
| Contractions | Custom `token_pattern` keeps `won't` whole | The default splits `won't` into `won`, which collides with the positive word "won". Expansion to `will not` stays in reserve if the coefficients justify it. |
| Lemmatization | None | Sentiment leans on surface forms. Matching `clean_text` at inference outweighs a smaller vocabulary. |
| Short reviews | Keep (no `word_count` filter) | You lose clear sentiment if you drop a three-word review like "complete waste money". |
| Duplicates | Keep, isolate via group-aware split | Dropping discards labeled data. A group split stops the same text from sitting in train and test at once. |
| `min_df` | 5 (vs. 10 in 03) | A lower floor admits rare but decisive words such as `refund` and `unplayable`. |
| Class imbalance | Stratify + `class_weight="balanced"` | Positive labels outnumber negative ones, about 69 to 31. Stratifying holds that ratio per fold; the weight lifts the minority class during fit. |
| Headline metric | F1 and PR-AUC, not accuracy | Accuracy rewards a model that always guesses positive. F1 and PR-AUC track the minority class. |

## 4. Group-Aware Train/Test Split

### 4.1 Derive group_id

The bool `duplicates` flag from 02 marks 'am I a non-first copy', which can't reconstruct which rows share a text (`keep = 'first'`). We rebuild the grouping the dedup implied: identical `review_clean` shares an id.

In [7]:

# pd.factorize assigns one integer per distinct text: identical -> same id,
# unique -> ist own. Collision-free by construction, unlike a hash

data['group_id'] = pd.factorize(data['review_clean'])[0]

n_groups = data['group_id'].nunique()
n_dupes = len(data) - n_groups

print(f'Rows:   {len(data):,}')
print(f'Groups: {n_groups:,} ')
print(f'Duplicate rows folded into groups: {n_dupes:,}')

Rows:   406,781
Groups: 356,718 
Duplicate rows folded into groups: 50,063


### 4.2 Split

sklearn has no StratifiedGroupShuffleSplit, so we take the first fold od a StratifiedGroupKFold as the hold-out test set.
n_splits = 5 leaves one fold out, which gives a ~80/20 split that keeps whole groups on one side an holds the 69/31 label ratio in both. 
GroupShuffleSplit would respect groups but skip stratification, risking a skewed test set under imbalance.

In [8]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits = N_SPLITS, shuffle = True,
                            random_state = RANDOM_STATE,)


train_idx, test_idx = next(
    sgkf.split(data['review_clean'], data['label'], groups = data['group_id'])
)

train = data.iloc[train_idx].reset_index(drop = True)
test = data.iloc[test_idx].reset_index(drop = True)

print(f'Train: {len(train):,}')
print(f'Test: {len(test):,}')

Train: 325,425
Test: 81,356


### 4.3 Verify

Two checks, one per rule: the assert proves no group straddles the split (integrity), and the balance print confirms the 69/31 ratio survived (stratification).

In [9]:
# Rule 1 - group integrity: no group_id may appeart on both sides.

overlap = set(train['group_id']) & set(test['group_id']) # set: [5, 5, 9, 2] -> [5, 9, 2]
assert not overlap, f'Leakage: {len(overlap)} groups in both train and test' # check if true or false
print('No group overlap between train and test')

# Rule 2 - stratification: positive-class share should match in both

print('\nPositive class share (label == 1):')
print(f'  full:   {data['label'].mean():,.3f}')
print(f'  train:  {train['label'].mean():,.3f}')
print(f'  test:   {test['label'].mean():,.3f}')

No group overlap between train and test

Positive class share (label == 1):
  full:   0.694
  train:  0.694
  test:   0.694


### 5. Vectorization (TF-IDF)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(**VECT_PARAMS)

# fit_transform on train set: learn vocab and IDF weights, then transform
X_train = vect.fit_transform(train['review_clean'])

# transform on test set: reuse train vocab/IDF, learn nothing new
X_test = vect.transform(test['review_clean'])


y_train = train['label']
y_test = test['label']

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Vocab size: {len(vect.get_feature_names_out()):,}')

X_train: (325425, 259426)
X_test: (81356, 259426)
Vocab size: 259,426


In [11]:
# token pattern check
vocab = set(vect.get_feature_names_out())
for w in ["won't", "don't", "not good", "not", "waste"]:
    print(f' {w!r:12} in vocab: {w in vocab}')

 "won't"      in vocab: True
 "don't"      in vocab: True
 'not good'   in vocab: True
 'not'        in vocab: True
 'waste'      in vocab: True


### 6. Baseline Model - Logistic Regresssions

In [12]:
from sklearn.linear_model import LogisticRegression

LOGREG_PARAMS = dict(
    class_weight = 'balanced',
    max_iter = 1000,
    random_state = RANDOM_STATE,
    n_jobs = 1,
)

logreg = LogisticRegression(**LOGREG_PARAMS)
logreg.fit(X_train, y_train)

print('Fitted...')

c:\Users\adris\Desktop\Data_Science_Projects\steam_reviews_nlp\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Fitted...


In [13]:
from sklearn.metrics import classification_report, f1_score, average_precision_score

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1] # P(class == 1)

# PR-AUC; average precision. Computed on the minority class label 0.

pr_auc_pos = average_precision_score(y_test, y_proba) # positive class
pr_auc_neg = average_precision_score(1 - y_test, 1 - y_proba) # negative class

print(f'PR-AUC (positive class): {pr_auc_pos:.3f}')
print(f'PR-AUC (negative class): {pr_auc_neg:.3f}')
print()
print(classification_report(y_test, y_pred, digits = 3))

PR-AUC (positive class): 0.977
PR-AUC (negative class): 0.894

              precision    recall  f1-score   support

           0      0.776     0.879     0.824     24860
           1      0.944     0.888     0.915     56496

    accuracy                          0.885     81356
   macro avg      0.860     0.884     0.870     81356
weighted avg      0.892     0.885     0.887     81356



## 7. Results — Logistic Regression baseline

Headline metrics on the hold-out test set (81,356 rows, 69/31 split):

| Metric | Positive (1) | Negative (0) |
|---|---|---|
| Precision | 0.944 | 0.776 |
| Recall | 0.888 | 0.879 |
| F1 | 0.915 | 0.824 |
| PR-AUC | 0.977 | 0.894 |

Accuracy 0.885, macro-F1 0.870. Accuracy is reported for context only; the
imbalance makes it a weak headline (a majority-only guess already scores 0.69).

**Reading the negative class (the hard one).** Recall 0.879 sits above
precision 0.776, which is the signature of `class_weight="balanced"`: the model
catches most true negative reviews and pays for it with more false alarms. This
is the intended trade-off from Section 3, visible in the numbers.

**PR-AUC 0.894 on the negative class** confirms the separation holds across all
thresholds, not just at the default 0.5 cut. Strong for a baseline.

**Baseline set.** LinearSVC and the tree models in Section 9 must beat
macro-F1 0.870 and negative-class PR-AUC 0.894 to earn their place.

### 8. Interpretability & Error Analysis

### 8.1 Overfitting Check

In [14]:
y_train_pred = logreg.predict(X_train)

train_f1_macro = f1_score(y_train, y_train_pred, average = 'macro')
test_f1_macro = f1_score(y_test, y_pred, average = 'macro')

print(f'Train macro-F1: {train_f1_macro:.3f}')
print(f'Test macro-F1: {test_f1_macro:.3f}')
print(f'Gap: {train_f1_macro - test_f1_macro:.3f}')

Train macro-F1: 0.893
Test macro-F1: 0.870
Gap: 0.023


### 8.2 Coefficient Analysis

In [15]:
coefs = pd.Series(
    logreg.coef_[0],
    index = vect.get_feature_names_out()
)

print('Top 20 features -> Negative (Not Recommended):')
print(coefs.sort_values().head(20).to_string())
print('\nTop 20 features -> Positive (Recommended):')
print(coefs.sort_values(ascending = False).head(20).to_string())

Top 20 features -> Negative (Not Recommended):
not worth         -11.993667
worst              -8.370606
refund             -7.992085
modding            -7.704157
unplayable         -7.186215
worse              -7.151274
not                -7.041395
mods               -6.825127
greedy             -6.568946
be fun             -6.513868
company            -6.485308
boring             -6.461011
horrible           -6.395709
dont buy           -6.351335
not recommend      -6.195525
can't recommend    -6.103134
terrible           -5.998210
trash              -5.942723
ruined             -5.851198
crashes            -5.805179

Top 20 features -> Positive (Recommended):
best          10.908297
amazing       10.300211
fun            9.266807
awesome        9.137158
is back        8.622180
love           8.218711
great          8.144113
addicting      7.725465
not bad        7.584753
10 10          7.526579
good           6.575756
fantastic      6.415182
perfect        6.159970
worth          6.

The top coefficients include game-identity terms, not pure sentiment:
`modding` (−7.70), `mods` (−6.83), and `soccer` (+5.48). These mirror the
vocabulary leakage documented in notebook 03: the model partly learns "reviews
about game X skew negative/positive" instead of sentiment language alone.
`modding`/`mods` ranking high suggests one game with mod-related complaints
carries many negative reviews.

**We keep these features and note the caveat, rather than filtering them.**
The reasoning:

1. It stays small. Only three or four of the top 20 features are domain terms.
   Everything else is clean sentiment (`not worth`, `refund`, `amazing`,
   `not bad`).
2. It fits the conservative variant. We avoid aggressive filtering here by
   design. Game-specific stopwords belonged to the aggressive variant in 03,
   and cutting them now would throw away signal that is sometimes generic.
3. On this corpus it is fair. For a sentiment model trained on this data,
   "this game tends to disappoint" is a reasonable thing to pick up on, not a
   mistake.

One caveat worth remembering: the model is tied to this particular mix of games.
On reviews from games it has never seen, these domain weights would not carry
over and could point the wrong way. Worth revisiting if a cross-game goal comes
up later.

### 8.3 Error Analysis

In [16]:
# Build a frame with truth, prediction, and the model's confidence 

errors = test[['review_clean', 'label']].copy()
errors['pred'] = y_pred
errors['proba'] = y_proba
errors = errors[errors['label'] != errors['pred']]  # keep only mistakes

# Two error types, sorted by how confident the model was in its wrong call:
# True negative review, model was sure it is a positive
fp = errors[errors['label'] == 0].sort_values('proba', ascending = False)

# True positive review, model was sure it is negative
fn = errors[errors['label'] == 1].sort_values('proba', ascending = True)

print(f'Confident false positives (neg review called positive): {len(fp):,}')
print(f"Confident false negatives (pos review called negative): {len(fn):,}\n")

for txt, p in fp.head(5)[['review_clean', 'proba']].values:
    print(f'[proba = {p:.2f}] {txt[:200]}')

Confident false positives (neg review called positive): 3,002
Confident false negatives (pos review called negative): 6,319

[proba = 1.00] definitely awesome
[proba = 1.00] awesome !!
[proba = 1.00] great game.
[proba = 1.00] great!
[proba = 1.00] best game 10/10


In [17]:
fp.head(5)

,review_clean,label,pred,proba
31575,definitely awesome,0,1,0.999924
1205,awesome !!,0,1,0.999918
53435,great game.,0,1,0.999786
53609,great!,0,1,0.999777
1654,best game 10/10,0,1,0.999678


In [18]:
for txt, p in fn.head(5)[['review_clean', 'proba']].values:
    print(f'[proba = {p:.2f}] {txt[:200]}')

[proba = 0.00] yet another update mods are still available and take two seem to have fully backed off. gta v can be recommended once more. )another update gamers kicked up enough of a stink that take two have retrac
[proba = 0.00] take two.
[proba = 0.00] owing to rockstar and take two now reneging on the modding ban i'm satisfied enough that this game and others in take two's stable are now worth buying again. that is so long as take two and 2k realis
[proba = 0.00] edit changed review to positive after rockstar acted as a mediator between their parent company and screwed over modders. take two remains a crap stain of a company but at least r* itself still has at
[proba = 0.00] not optimaz game


In [19]:
fn.head(5)

,review_clean,label,pred,proba
36308,yet another update mods are still available an...,1,0,0.000334
35363,take two.,1,0,0.000848
26574,owing to rockstar and take two now reneging on...,1,0,0.001200
35171,edit changed review to positive after rockstar...,1,0,0.001335
48475,not optimaz game,1,0,0.001425


Inspecting the most confident mistakes (proba near 0 and 1) split the errors
into two distinct causes, one per direction.

**Confident false positives (3,002) — mostly mislabeled data, not a model error.**
The top cases are short, clearly positive texts marked as Not Recommended:
"definitely awesome", "great game.", "best game 10/10". The model reads them as
positive and is right; the label is wrong. This fits how Steam labels work: the
label is the user's Recommend button, which can contradict the written text
(misclick, protest, or sarcasm toward something else). Implication: part of the
error rate is label noise, so the model's true quality sits somewhat above the
0.870 macro-F1. The ceiling is capped by the data, not the model.

**Confident false negatives (6,319) — mostly vocabulary leakage.**
The top cases are genuinely positive reviews the model calls negative with full
confidence. They share the same terms: `mods`, `modding`, `take two`,
`rockstar`. These are the strongest non-sentiment negative coefficients from
Section 8.1. The reviews discuss a modding controversy that ended well ("can be
recommended once more"), but the model reads the domain terms instead of the
sentiment and predicts negative. This is the leakage documented in the earlier
DECISION block, now visible as concrete lost predictions.

**Takeaway.** The false positives suggest the headline metric understates the
model. The false negatives confirm the leakage trade-off has a real cost, which
strengthens the case for revisiting it if a cross-game goal appears. For the
baseline, both findings stand as documented, not treated.

### 9. Stronger Models

### 9.1 LinearSVC

In [20]:
from sklearn.svm import LinearSVC


SVC_PARAMS = dict(
    class_weight = 'balanced',
    dual = 'auto',
    random_state = RANDOM_STATE,
)

svc = LinearSVC(**SVC_PARAMS)
svc.fit(X_train, y_train)

y_pred_svc = svc.predict(X_test)

y_score_svc = svc.decision_function(X_test)

print('LinearSVC')
print(f'macro-F1:               {f1_score(y_test, y_pred_svc, average = 'macro'):.3f}')
print(f'PR-AUC (neg class):     {average_precision_score(1 - y_test, -y_score_svc):.3f}') # flip labels and scores
print(f'PR-AUC (pos class):     {average_precision_score(y_test, y_score_svc):.3f}')
print(classification_report(y_test, y_pred_svc, digits = 3))

LinearSVC
macro-F1:               0.868
PR-AUC (neg class):     0.891
PR-AUC (pos class):     0.976
              precision    recall  f1-score   support

           0      0.789     0.852     0.819     24860
           1      0.933     0.900     0.916     56496

    accuracy                          0.885     81356
   macro avg      0.861     0.876     0.868     81356
weighted avg      0.889     0.885     0.886     81356



macro-F1 0.868, PR-AUC neg 0.891 — within noise of the LogReg baseline
(0.870 / 0.894).<br> 

Same linear geometry on the same sparse TF-IDF, so the
margin-vs-probability difference does not matter here.<br> 

LogReg stays the pick because it offers calibrated `predict_proba`; LinearSVC only gives
`decision_function` scores.

### 9.2 XGBoost 

#### 9.2.1

In [21]:
from xgboost import XGBClassifier

XGB_DIR = '../models/sentiment'
XGB_PATH = os.path.join(XGB_DIR, 'xgb_tfidf.joblib')

In [25]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

scale = n_pos/n_neg
print('scale_pos_weight:', round(scale, 3))

scale_pos_weight: 2.273


In [26]:
XGB_PARAMS = dict(
    n_estimators = 300,
    max_depth = 6,
    tree_method = 'hist',
    scale_pos_weight = scale,
    n_jobs = -1,
    random_state = RANDOM_STATE,
    eval_metric = 'logloss',
)

In [27]:
if os.path.exists(XGB_PATH):
    xgb = joblib.load(XGB_PATH)
    print('Loaded cached XGBoost model')
else:
    xgb = XGBClassifier(**XGB_PARAMS)
    xgb.fit(X_train, y_train)
    joblib.dump(xgb, XGB_PATH)
    print('Trained and cached XGBoost model.')

Trained and cached XGBoost model.


In [29]:
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]


print('XGBoostClassifier')
print(f'macro-F1:               {f1_score(y_test, y_pred_xgb, average = 'macro'):.3f}')
print(f'PR-AUC (neg class):     {average_precision_score(1 - y_test, -y_proba_xgb):.3f}') # flip labels and scores
print(f'PR-AUC (pos class):     {average_precision_score(y_test, y_proba_xgb):.3f}')
print()
print(classification_report(y_test, y_pred_xgb, digits = 3))

XGBoostClassifier
macro-F1:               0.807
PR-AUC (neg class):     0.877
PR-AUC (pos class):     0.971

              precision    recall  f1-score   support

           0      0.902     0.588     0.712     24860
           1      0.843     0.972     0.903     56496

    accuracy                          0.854     81356
   macro avg      0.872     0.780     0.807     81356
weighted avg      0.861     0.854     0.844     81356



In [30]:
# Same overfitting check as the baseline: train vs. test macro-F1

y_train_pred_xgb = xgb.predict(X_train)

train_f1 = f1_score(y_train, y_train_pred_xgb, average="macro")
test_f1  = f1_score(y_test,  y_pred_xgb,       average="macro")

print(f"Train macro-F1: {train_f1:.3f}")
print(f"Test  macro-F1: {test_f1:.3f}")
print(f"Gap:            {train_f1 - test_f1:.3f}")

Train macro-F1: 0.841
Test  macro-F1: 0.807
Gap:            0.034


macro-F1 0.807, PR-AUC neg 0.877 — clearly below the LogReg baseline
(0.870 / 0.894), despite a heavier model and correct imbalance weighting
(scale_pos_weight 2.273).

The negative class is where it breaks: precision 0.902 but recall 0.588, so the
model catches only 59% of true negative reviews. It predicts "negative" only
when very sure and lets the rest pass as positive (class-1 recall 0.972). This
is the opposite of the baseline, where balanced weighting pushed recall above
precision on class 0.

Fitting check: train macro-F1 0.841, test 0.807, gap 0.034. The small gap rules
out overfitting; the modest train score itself points to mild underfitting.
Capacity is not the problem (300 trees, depth 6 is plenty) — the sparse TF-IDF
representation does not offer trees the splits they need. More depth or trees
would widen the gap without lifting the train ceiling.

Takeaway: match the model to the feature geometry. A tree cannot sum many weak
n-gram signals the way a linear model does, and that is exactly what separates
negative reviews here.

#### 9.2.2

In [33]:
from scipy.sparse import hstack

# Pull clean metadata columns (helpful excluded)
meta_cols = ['hour_played', 'is_early_access_review']
meta_train = train[meta_cols].astype(float).values
meta_test = test[meta_cols].astype(float).values


# Glue metadata onto the TF-IDF matrix (just columns, not rows)
X_train_combined = hstack([X_train, meta_train]).tocsr()
X_test_combined = hstack([X_test, meta_test]).tocsr()

print(f'TF-IDF only:    {X_train.shape}')
print(f'Combined:       {X_train_combined.shape} (+ {len(meta_cols)} metadata features)')

TF-IDF only:    (325425, 259426)
Combined:       (325425, 259428) (+ 2 metadata features)


In [34]:
# Train combined matrix and cache under new file name

XGB_META_PATH = os.path.join(MODELS_DIR, 'xgb_meta.joblib')

if os.path.exists(XGB_META_PATH):
    xgb_meta = joblib.load(XGB_META_PATH)
    print(f'Loaded cached XGBoost (meta) model')

else:
    xgb_meta = XGBClassifier(**XGB_PARAMS)
    xgb_meta.fit(X_train_combined, y_train)
    joblib.dump(xgb_meta, XGB_META_PATH)
    print('Trained and cached XGBoost (meta) model')

Trained and cached XGBoost (meta) model


In [35]:
# Evaluate with same metrics

y_pred_meta = xgb_meta.predict(X_test_combined)
y_proba_meta = xgb_meta.predict_proba(X_test_combined)[:, 1]

print('XGBoost (TF-IDF + metadata)')
print(f'macro-F1:               {f1_score(y_test, y_pred_meta, average = 'macro'):.3f}')
print(f'PR-AUC (neg class):     {average_precision_score(1 - y_test, -y_proba_meta):.3f}') # flip labels and scores
print(f'PR-AUC (pos class):     {average_precision_score(y_test, y_proba_meta):.3f}')
print()
print(classification_report(y_test, y_pred_meta, digits = 3))

XGBoost (TF-IDF + metadata)
macro-F1:               0.814
PR-AUC (neg class):     0.883
PR-AUC (pos class):     0.972

              precision    recall  f1-score   support

           0      0.902     0.602     0.722     24860
           1      0.847     0.971     0.905     56496

    accuracy                          0.858     81356
   macro avg      0.875     0.787     0.814     81356
weighted avg      0.864     0.858     0.849     81356



In [37]:
# Feature Importance 

feat_names = list(vect.get_feature_names_out()) + meta_cols
importances = pd.Series(xgb_meta.feature_importances_, index = feat_names)

print('Where the metadata feature rank:')
for c in meta_cols:
    rank = importances.rank(ascending = False)[c]
    print(f' {c:24} importances = {importances[c]:.4f}  rank {int(rank):,}')


print('\nTop 15 features overall:')
print(importances.sort_values(ascending = False).head(15).to_string())

Where the metadata feature rank:
 hour_played              importances = 0.0005  rank 591
 is_early_access_review   importances = 0.0007  rank 325

Top 15 features overall:
take two      0.011508
not buy       0.010969
money         0.006828
best          0.006672
dont buy      0.006289
waste         0.005955
garbage       0.005760
awesome       0.005249
trash         0.005027
modding       0.005010
piece of      0.004974
and lovers    0.004952
nice          0.004756
10 10         0.004695
not worth     0.004625


macro-F1 0.814, PR-AUC neg 0.883 — a marginal lift over TF-IDF-only XGBoost
(0.807 / 0.877), still below the LogReg baseline (0.870 / 0.894).

The metadata carries almost no weight. `hour_played` ranks 591st and
`is_early_access_review` 325th, both with importance near 0.0005. The model saw
them and judged them near-useless. The top 15 features are all text
(`take two`, `not buy`, `waste`). For sentiment on these reviews, the signal
sits in the words; play time and early-access status add little the text does
not already reveal.

`helpful` was excluded on purpose: its votes accrue after posting, so using it
would leak future information a fresh review would not have.

Conclusion: metadata does not rescue the tree model. The class-0 recall problem
(0.602) is untouched, because it stems from feature geometry, not missing
features.

### 9.3 Model comparison

| Model | macro-F1 | PR-AUC (neg) | Note |
|---|---|---|---|
| Logistic Regression | **0.870** | **0.894** | baseline; calibrated probabilities |
| LinearSVC | 0.868 | 0.891 | within noise of LogReg; no probabilities |
| XGBoost (TF-IDF) | 0.807 | 0.877 | class-0 recall collapses (0.588) |
| XGBoost (+ metadata) | 0.814 | 0.883 | metadata adds almost nothing |

**Winner: Logistic Regression.** The two linear models tie within noise, and
LogReg wins on a tiebreaker unrelated to score — it gives calibrated
`predict_proba`, useful for threshold tuning and the topic-sentiment join ahead.
Both tree variants fall short: sparse TF-IDF does not give trees the splits they
need, and the metadata carries no signal the text does not already hold.

### 10. Combined Insights - Topics & Sentiment

### 11. Persist Artifacts

### 12. Summary